# 前処理

### 成果物

preprocessed.csv … 前処理完了データ<br>
　　　　※train/test複合、Suvived/Name列削除)<br>
Suvived.csv … trainから抽出したSuvived列

In [1]:
#各種インポート
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [2]:
#モデル作成用データの取得
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
print(train.shape)

(891, 12)


In [4]:
print(test.shape)

(418, 11)


In [5]:
#前処理用データフレーム
pre_data = pd.concat([train, test], ignore_index=True)
pre_data = pre_data.drop(columns="Survived")
pre_data.to_csv("pre_data.csv", index=False)

survived = train["Survived"]
survived.to_csv("survived.csv", index=False)

In [6]:
print(pre_data.shape)

(1309, 11)


In [7]:
#欠損の補完(Age)
imputer = SimpleImputer(strategy="mean")
pre_data[["Age"]] = imputer.fit_transform(pre_data[["Age"]])

In [8]:
#チケット番号の補正
split_data = pre_data["Ticket"].str.split(r"\s(\d{3,})",expand=True)
split_data = split_data.drop(columns=[2])
split_data[1] = split_data[1].fillna(split_data[0])
for i in range(1, len(split_data)):
    if split_data.loc[i, 0].isdigit():
        split_data.loc[i, 0] = None if split_data.loc[i-1, 0].isdigit() else split_data.loc[i-1, 0]

In [9]:
#Ticket_IDだけでNoがない箇所をNoneへ変換して補完
repl_dict = {
    "LINE": None,
    "S.O./P.P. 2": None,
    "S.O./P.P. 3": None
}
split_data[1] = split_data[1].replace(repl_dict)

split_df = pd.DataFrame(split_data[1])
split_df = pd.DataFrame(imputer.fit_transform(split_df), columns=[1])

In [10]:
split_df.to_csv("split_data.csv", index=False)

In [11]:
split_data[2] = split_df
split_data = split_data.drop(columns=[1])

In [12]:
#Ticket_IDの似てる表現をまとめる
from difflib import SequenceMatcher

threshold = 0.8

groups = []
for index, value in enumerate(split_data[0]):
    found = False
    for group in groups:
        if SequenceMatcher(None, value, group[0]).ratio() > threshold:
            group.append(value)
            found = True
            break
    if not found:
        groups.append([value])

uni_value = {}
for i, group in enumerate(groups):
    uni_value[i] = set(group)

In [13]:
#似た表現があるのはuni_value1へ　ないのはuni_value2へ
uni_value1 = {}
uni_value2 = {}
a = 0
b = 0
for key,value in uni_value.items():
    if len(value) > 1:
        uni_value1[a] = value
        a = a + 1
    else:
        uni_value2[b] = value
        b = b + 1

In [14]:
#見やすくするためにuni_value1をcsvへ書き出す
max_length = max(len(s) for s in uni_value1.values())
adjusted_sets = {k: list(s) + [''] * (max_length - len(s)) for k, s in uni_value1.items()}
df = pd.DataFrame.from_dict(adjusted_sets)
df.to_csv("uni_value.csv", index=False)

In [15]:
#置換辞書
repl_dict2 = {
    "A/5": "A/5.",
    "A./5.": "A/5.",
    "A.5.": "A/5.",
    "STON/O 2.": "SOTON/O2.",
    "SOTON/O2": "SOTON/O2.",
    "SOTON/O2": "SOTON/O2.",
    "CA.": "C.A.",
    "A4.": "A/4.",
    "A/4": "A/4.",
    "S.O./P.P. 2": "S.O./P.P.",
    "S.O.C.": "S.O./P.P.",
    "S.O.P.": "S.O./P.P.",
    "STON/OQ.": "SOTON/OQ.",
    "SOTON/O.Q.": "SOTON/OQ.",
    "SOTON/OQ": "SOTON/OQ.",
    "SC/Paris": "SC/Paris.",
    "S.C./PARIS": "SC/Paris.",
    "SC/PARIS": "SC/Paris.",
    "SW/PP": "S.W./PP.",
    "S.W./PP": "S.W./PP.",
    "S.O./P.P. 3": "S.O./P.P."
}
split_data[0] = split_data[0].replace(repl_dict2)

In [16]:
#IDとNoに分割したTicketを追加して元のTicketを削除
pre_data[["Ticket_ID","Ticket_No"]] = split_data
pre_data.drop(columns=["Ticket"], inplace=True)

In [17]:
#確認のために書き出し
pre_data.to_csv("pre_data2.csv",index=False)

In [18]:
#Cabinの回帰補完(前処理:いらない列を落としてデータ有無で分割)
cabin_Data = pre_data.drop(columns=["Name", "Sex", "Age", "Embarked", "Ticket_ID"])
cabin_Data[["Fare"]] = imputer.fit_transform(cabin_Data[["Fare"]])

cabin_train = pd.DataFrame(columns=cabin_Data.columns)
cabin_test = pd.DataFrame(columns=cabin_Data.columns)

for index, row in cabin_Data.iterrows():
    if pd.notna(row["Cabin"]):
        cabin_train = pd.concat([cabin_train, row.to_frame().T])
    else:
        cabin_test = pd.concat([cabin_test, row.to_frame().T])

cabin_train["Cabin"] = cabin_train["Cabin"].astype(str).str[0]

cabin_X = cabin_train.drop(columns="Cabin")
cabin_y = cabin_train["Cabin"]

cabin_test_X = cabin_test.drop(columns="Cabin")

In [19]:
print(cabin_train.shape)
print(cabin_test.shape)

(295, 7)
(1014, 7)


In [20]:
#Cabinの回帰補完
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

model = LogisticRegression(max_iter=1000)

In [21]:
#パラメータ最適化&クロスバリデーション
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
              'penalty': ['l2']}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring="accuracy")
grid_search.fit(cabin_X, cabin_y)

best_model = grid_search.best_estimator_

C:\Users\命燐\Python Practice\TestPy\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
C:\Users\命燐\Python Practice\TestPy\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\命燐\Python Practice\TestPy\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
 

In [22]:
#予測して結果をcabin_testに格納
cabin_test["Cabin"] = best_model.predict(cabin_test_X)

In [23]:
#確認のために書き出し
cabin_test.to_csv("cabin_test.csv", index=False)

In [24]:
#結合してソートし、元のデータにCabinを入れる
cabin_Data2 = pd.concat([cabin_train, cabin_test])
cabin_Data2 = cabin_Data2.sort_values(by="PassengerId")

pre_data["Cabin"] = cabin_Data2["Cabin"]

In [25]:
#Embarkedの前処理:いらない列を落としてデータ有無で分割
embarked_Data = pre_data.drop(columns=["Name", "Sex", "Age", "SibSp", "Parch", "Cabin", "Ticket_ID"])
embarked_Data[["Fare"]] = imputer.fit_transform(embarked_Data[["Fare"]])

embarked_train = pd.DataFrame(columns=embarked_Data.columns)
embarked_test = pd.DataFrame(columns=embarked_Data.columns)

for index, row in embarked_Data.iterrows():
    if pd.notna(row["Embarked"]):
        embarked_train = pd.concat([embarked_train, row.to_frame().T])
    else:
        embarked_test = pd.concat([embarked_test, row.to_frame().T])

embarked_X = embarked_train.drop(columns="Embarked")
embarked_y = embarked_train["Embarked"]

embarked_test_X = embarked_test.drop(columns="Embarked")

In [26]:
print(embarked_train.shape)
print(embarked_test.shape)

(1307, 5)
(2, 5)


In [27]:
#Embarkedの回帰補完
model2 = LogisticRegression(max_iter=5000)

In [28]:
#パラメータ最適化&クロスバリデーション
param_grid2 = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
              'penalty': ['l2']}

grid_search2 = GridSearchCV(estimator=model, param_grid=param_grid2, cv=5, scoring="accuracy")
grid_search2.fit(embarked_X, embarked_y)

best_model2 = grid_search2.best_estimator_

C:\Users\命燐\Python Practice\TestPy\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\命燐\Python Practice\TestPy\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-

In [29]:
#予測して結果をembarked_testに格納
embarked_test["Embarked"] = best_model2.predict(embarked_test_X)

In [30]:
#結合してソートし、元のデータにEmbarkedを入れる
embarked_Data2 = pd.concat([embarked_train, embarked_test])
embarked_Data2 = embarked_Data2.sort_values(by="PassengerId")

pre_data["Embarked"] = embarked_Data2["Embarked"]

pre_data[["Fare"]] = imputer.fit_transform(pre_data[["Fare"]])

#不要なNameを削除
pre_data = pre_data.drop(columns="Name")

#Sexの置換(♂=0,♀=1)
repl_dict3 = {
    "male": "0",
    "female": "1"
}
pre_data["Sex"] = pre_data["Sex"].replace(repl_dict3)

#後続用csvファイルに補完
pre_data.to_csv("preprocessed.csv", index=False)